# Загрузка и распаковка данных

In [1]:
pip install geopandas shapely pyogrio fiona unidecode tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path

In [ ]:
# Путь к папке с ZIP-файлами
INPUT_DIR = Path(r"D:\Users\smeen\Downloads\ДТП")

# Папка для распаковки
EXTRACT_DIR = INPUT_DIR / "_extracted"

# Папка для итоговых файлов
OUTPUT_DIR = INPUT_DIR / "_output"

# Создаём папки, если их нет
EXTRACT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

print("INPUT_DIR:", INPUT_DIR)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

INPUT_DIR: D:\Users\smeen\Downloads\ДТП
EXTRACT_DIR: D:\Users\smeen\Downloads\ДТП\_extracted
OUTPUT_DIR: D:\Users\smeen\Downloads\ДТП\_output


In [ ]:
import os
import re
import json
import zipfile
import warnings
from typing import List, Optional
import pandas as pd
from tqdm import tqdm

try:
    import geopandas as gpd
    from shapely.geometry import shape
    GEOPANDAS_OK = True
except:
    GEOPANDAS_OK = False

from unidecode import unidecode


# Функция нормализации имён столбцов
def normalize_colname(name: str) -> str:
    name = unidecode(str(name))
    name = re.sub(r'[^0-9a-zA-Z_]+', '_', name).strip('_')
    return name.lower()


# Функция распаковки всех ZIP
def unzip_all_zips(input_dir: Path, extract_dir: Path) -> List[Path]:
    extracted = []
    zips = list(input_dir.rglob("*.zip"))
    print(f"Найдено ZIP-файлов: {len(zips)}")
    for zpath in tqdm(zips, desc="Распаковка"):
        with zipfile.ZipFile(zpath, 'r') as zf:
            folder = extract_dir / zpath.stem
            folder.mkdir(exist_ok=True)
            zf.extractall(folder)
            for info in zf.infolist():
                if not info.is_dir():
                    extracted.append(folder / info.filename)
    return extracted


# Поиск всех .geojson файлов
def find_geojsons(root: Path) -> List[Path]:
    files = list(root.rglob("*.geojson"))
    print(f"Найдено .geojson файлов: {len(files)}")
    return files

In [ ]:
_ = unzip_all_zips(INPUT_DIR, EXTRACT_DIR)

Найдено ZIP-файлов: 82


Распаковка: 100%|██████████| 82/82 [00:48<00:00,  1.70it/s]


In [ ]:
geojson_files = find_geojsons(EXTRACT_DIR)
print(f"Всего найдено файлов: {len(geojson_files)}")

Найдено .geojson файлов: 164
Всего найдено файлов: 164


In [ ]:
frames = []

print("Загрузка данных...")
for p in tqdm(geojson_files):
    try:
        if GEOPANDAS_OK:
            gdf = gpd.read_file(p)
            gdf["source_file"] = str(p)
            frames.append(gdf)
        else:
            with open(p, "r", encoding="utf-8") as f:
                data = json.load(f)
                props = [feat["properties"] for feat in data["features"]]
                df = pd.DataFrame(props)
                df["source_file"] = str(p)
                frames.append(df)
    except Exception as e:
        warnings.warn(f"Ошибка при загрузке {p}: {e}")

combined = pd.concat(frames, ignore_index=True)
print(f"Размер объединённого файла: {combined.shape}")

Загрузка данных...


  0%|          | 0/164 [00:00<?, ?it/s]C:\Users\smeen\AppData\Local\Temp\ipykernel_14012\3124203366.py:18: UserWarning: Ошибка при загрузке D:\Users\smeen\Downloads\ДТП\_extracted\amurskaia-oblast.geojson: D:\Users\smeen\Downloads\ДТП\_extracted\amurskaia-oblast.geojson: Permission denied
  warnings.warn(f"Ошибка при загрузке {p}: {e}")
  1%|          | 1/164 [00:01<03:17,  1.21s/it]C:\Users\smeen\AppData\Local\Temp\ipykernel_14012\3124203366.py:18: UserWarning: Ошибка при загрузке D:\Users\smeen\Downloads\ДТП\_extracted\arkhangelskaia-oblast.geojson: D:\Users\smeen\Downloads\ДТП\_extracted\arkhangelskaia-oblast.geojson: Permission denied
  warnings.warn(f"Ошибка при загрузке {p}: {e}")
C:\Users\smeen\AppData\Local\Temp\ipykernel_14012\3124203366.py:18: UserWarning: Ошибка при загрузке D:\Users\smeen\Downloads\ДТП\_extracted\astrakhanskaia-oblast.geojson: D:\Users\smeen\Downloads\ДТП\_extracted\astrakhanskaia-oblast.geojson: Permission denied
  warnings.warn(f"Ошибка при загрузке {p}: 

Размер объединённого файла: (1465882, 18)


In [ ]:
csv_path = OUTPUT_DIR / "dtp_combined.csv"
combined.to_csv(csv_path, index=False, encoding="utf-8")
print(f"Сохранено в: {csv_path}")

Сохранено в: D:\Users\smeen\Downloads\ДТП\_output\dtp_combined.csv


In [ ]:
combined.head()

,id,light,point,region,scheme,address,category,datetime,severity,vehicles,dead_count,participants,injured_count,parent_region,participants_count,geometry,source_file,nearby
0,189548,Светлое время суток,"{ ""lat"": 49.789873999999998, ""long"": 129.84691...",Бурейский район,830,"Обход п. Бурея, 5 км",Наезд на пешехода,2019-04-05 14:10:00,Легкий,"[ { ""year"": 2005, ""brand"": ""TOYOTA"", ""color"": ...",0,"[ { ""role"": ""Пешеход"", ""gender"": ""Мужской"", ""v...",1,Амурская область,2,POINT (129.84692 49.78987),D:\Users\smeen\Downloads\ДТП\_extracted\amursk...,NaN
1,188700,"В темное время суток, освещение отсутствует","{ ""lat"": 49.755600000000001, ""long"": 129.29220...",Бурейский район,200,"Благовещенск – Гомелевка, 148 км",Столкновение,2022-11-16 22:50:00,С погибшими,"[ { ""year"": 1993, ""brand"": ""TOYOTA"", ""color"": ...",1,[ ],1,Амурская область,3,POINT (129.2922 49.7556),D:\Users\smeen\Downloads\ДТП\_extracted\amursk...,NaN
2,188702,"В темное время суток, освещение не включено","{ ""lat"": 49.787187000000003, ""long"": 129.81648...",Бурейский район,600,"Обход п. Бурея, 2 км",Съезд с дороги,2019-12-27 23:50:00,Легкий,"[ { ""year"": 2008, ""brand"": ""TOYOTA"", ""color"": ...",0,[ ],1,Амурская область,1,POINT (129.81648 49.78719),D:\Users\smeen\Downloads\ДТП\_extracted\amursk...,NaN
3,188704,"В темное время суток, освещение отсутствует","{ ""lat"": 49.135599999999997, ""long"": 129.1181 }",Бурейский район,830,"пгт Бурея, ул Райчихинская, 1",Наезд на пешехода,2016-12-30 19:10:00,С погибшими,"[ { ""year"": 1989, ""brand"": ""TOYOTA"", ""color"": ...",1,"[ { ""role"": ""Пешеход"", ""gender"": ""Мужской"", ""v...",0,Амурская область,2,POINT (129.1181 49.1356),D:\Users\smeen\Downloads\ДТП\_extracted\amursk...,NaN
4,188706,Светлое время суток,"{ ""lat"": 49.119199999999999, ""long"": 129.1353 }",Бурейский район,730,"пгт Новобурейский, ул Лесная, 22",Наезд на пешехода,2017-12-30 08:45:00,Легкий,"[ { ""year"": 2003, ""brand"": ""TOYOTA"", ""color"": ...",0,"[ { ""role"": ""Пешеход"", ""gender"": ""Женский"", ""v...",1,Амурская область,2,POINT (129.1353 49.1192),D:\Users\smeen\Downloads\ДТП\_extracted\amursk...,NaN
